In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Activation,  Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
# from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.utils import plot_model
from scipy import signal
from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
import keras_tuner as kt
import keras_lmu
from sklearn.model_selection import train_test_split
#import torch
import tensorflow.keras as keras
import tensorflow.keras.backend as keras_backend
import second_gen_RNN
import third_gen_RLIF
import random
from Quantized_HSNN import GSN
%matplotlib qt

In [ ]:
meta_loss_RNN  = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\meta_loss_RNN_dropout_10_percent.csv", delimiter=',')
meta_loss_SNN  = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\meta_loss_SNN.csv", delimiter=',')
meta_loss_HSNN = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\meta_loss_HSNN.csv", delimiter=',')
meta_loss_RNN_no_dropout  = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\meta_loss_RNN_no_dropout.csv", delimiter=',')

In [ ]:
test_loss_RNN  = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\test_loss_RNN_dropout_10_percent.csv", delimiter=',')
test_loss_SNN  = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\test_loss_SNN.csv", delimiter=',')
test_loss_HSNN = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\val_loss_HSNN.csv", delimiter=',')
test_loss_RNN_no_dropout = np.genfromtxt(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Loss_info_SNN_HSNN_RNN\test_loss_RNN_no_dropout.csv", delimiter=',')

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# Define figure size and DPI for high-quality output
fig, ax = plt.subplots(figsize=(8, 6))

# Main plots with distinct markers
ax.plot(meta_loss_RNN[0:100], label='Meta loss RNN with dropout', linestyle='-', marker='o', markersize=5, linewidth=2, color='#1f77b4')
ax.plot(meta_loss_HSNN[0:100], label='Meta loss HSNN', linestyle='--', marker='s', markersize=5, linewidth=2, color='#ff7f0e')
ax.plot(meta_loss_SNN[0:100], label='Meta loss SNN', linestyle='-.', marker='d', markersize=5, linewidth=2, color='#2ca02c')
ax.plot(meta_loss_RNN_no_dropout[0:100], label='Meta loss RNN without dropout', linestyle='-', marker='x', markersize=5, linewidth=2, color='#d62728')

# Add main labels
ax.set_xlabel('Epochs', fontsize=14)
ax.set_ylabel('Loss', fontsize=14)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
ax.grid(True, linestyle='--', alpha=0.6)
#ax.legend(fontsize=12, loc='upper left', bbox_to_anchor=(1, 1))

# Create inset zoomed-in area
axins = inset_axes(ax, width="45%", height="45%", loc='center right')  # Adjust size and position

# Plot same data but zoomed into a specific range
zoom_range = slice(39, 59)  # Change this to focus on a specific region
axins.set_xlim(zoom_range.start, zoom_range.stop)
axins.plot(range(39, 59), meta_loss_RNN[zoom_range], linestyle='-', marker='o', markersize=5, linewidth=2, color='#1f77b4')
axins.plot(range(39, 59), meta_loss_HSNN[zoom_range], linestyle='--', marker='s', markersize=5, linewidth=2, color='#ff7f0e')
axins.plot(range(39, 59), meta_loss_SNN[zoom_range], linestyle='-.', marker='d', markersize=5, linewidth=2, color='#2ca02c')
axins.plot(range(39, 59), meta_loss_RNN_no_dropout[zoom_range], linestyle='-.', marker='d', markersize=5, linewidth=2, color='#d62728')


#axins1 = inset_axes(ax, width="40%", height="40%", loc='lower right')  # Adjust size and position

# Plot same data but zoomed into a specific range
# zoom_range1 = slice(70, 100)  # Change this to focus on a specific region
# axins1.plot(meta_loss_RNN[zoom_range1], linestyle='-', marker='o', markersize=5, linewidth=2, color='#1f77b4')
# axins1.plot(meta_loss_HSNN[zoom_range1], linestyle='--', marker='s', markersize=5, linewidth=2, color='#ff7f0e')
# axins1.plot(meta_loss_SNN[zoom_range1], linestyle='-.', marker='d', markersize=5, linewidth=2, color='#2ca02c')

# Set limits for zoomed-in plot
#axins.set_xlim(zoom_range.start, zoom_range.stop)
# axins.set_ylim(min(min(meta_loss_RNN[zoom_range]), min(meta_loss_HSNN[zoom_range]), min(meta_loss_SNN[zoom_range])) * 0.95, 
#                max(max(meta_loss_RNN[zoom_range]), max(meta_loss_HSNN[zoom_range]), max(meta_loss_SNN[zoom_range])) * 1.05)  # Auto-adjusted y-axis

# axins.set_ylim(0, max(val_loss_HSNN_ANN[zoom_range]))  # Adjust based on your data
# axins.set_xticks([])  # Remove x-ticks for clarity
# axins.set_yticks([])  # Remove y-ticks for clarity

# Add a rectangle in the main plot showing the zoomed area
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="black", linestyle="--")

# Final adjustments
#plt.tight_layout()
ax.legend(fontsize=12, loc='best', frameon=True)

plt.show()
plt.savefig(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Plots\Meta_loss.pdf", format="pdf", bbox_inches="tight")
plt.savefig(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Plots\Meta_loss.svg", format="svg", bbox_inches="tight")




In [ ]:
meta_loss_HSNN[-1]

In [ ]:
np.sqrt(0.00025838325)

In [ ]:
import numpy as np

def quantize_weights(weights, num_bits=8):
    """
    Quantizes the given weights to the specified bit-width.

    Parameters:
    - weights: np.ndarray, original floating-point weights.
    - num_bits: int, bit-width for quantization (e.g., 8 or 16).

    Returns:
    - quantized_weights: np.ndarray, quantized integer weights.
    - scale: float, scaling factor used for quantization.
    """
    # Determine the maximum representable value based on num_bits
    max_int = 2**(num_bits - 1) - 1

    # Compute scaling factor
    max_weight = np.max(np.abs(weights))
    scale = max_int / max_weight if max_weight != 0 else 1

    # Quantize weights
    quantized_weights = np.round(weights * scale).astype(np.int32)

    # Clip to ensure within representable range
    quantized_weights = np.clip(quantized_weights, -max_int, max_int)

    return quantized_weights, scale

# Example usage:
original_weights = np.array([0.1, -0.5, 0.3, -0.8], dtype=np.float32)
quantized_weights, scale = quantize_weights(original_weights, num_bits=8)
print("Quantized Weights:", quantized_weights)
print("Scaling Factor:", scale)



In [2]:
cell = GSN(1100)

quantized_layer = tf.keras.layers.RNN(cell)

In [5]:
quantized_layer(tf.random.uniform(shape=(1,1,5))).numpy().max()

0.0

In [ ]:
quantized_layer.get_weights()